In [ ]:
!unzip "/content/drive/MyDrive/SPIDER/spider_data (1).zip" -d spider

Archive:  /content/drive/MyDrive/SPIDER/spider_data (1).zip
   creating: spider/spider_data/
  inflating: spider/spider_data/dev_gold.sql  
  inflating: spider/__MACOSX/spider_data/._dev_gold.sql  
   creating: spider/spider_data/database/
  inflating: spider/__MACOSX/spider_data/._database  
  inflating: spider/spider_data/.DS_Store  
  inflating: spider/__MACOSX/spider_data/._.DS_Store  
  inflating: spider/spider_data/test_tables.json  
  inflating: spider/__MACOSX/spider_data/._test_tables.json  
  inflating: spider/spider_data/train_others.json  
  inflating: spider/__MACOSX/spider_data/._train_others.json  
  inflating: spider/spider_data/train_spider.json  
  inflating: spider/__MACOSX/spider_data/._train_spider.json  
  inflating: spider/spider_data/test.json  
  inflating: spider/__MACOSX/spider_data/._test.json  
  inflating: spider/spider_data/tables.json  
  inflating: spider/__MACOSX/spider_data/._tables.json  
  inflating: spider/spider_data/dev.json  
  inflating: spider

EDA

In [ ]:
import os
import json
import random
import logging
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

@dataclass
class Config:
    data_dir: str = "./spider/spider_data"
    output_dir: str = "./artifacts"
    model_dir: str = "./artifacts/models"
    test_size: float = 0.2
    val_size: float = 0.25
    max_length: int = 128
    sentence_model: str = "all-MiniLM-L6-v2"
    bert_model: str = "distilbert-base-uncased"
    train_epochs: int = 5
    lr_bert: float = 2e-5
    batch_size: int = 16
    weight_decay: float = 0.01
    lr_max_iter: int = 1000
    lr_c: float = 1.0

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.model_dir, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(cfg.output_dir, "experiment.log")),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

with open(os.path.join(cfg.output_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

In [ ]:
def find_sqlite_path(db_dir: str, db_id: str) -> Optional[str]:
    folder = os.path.join(db_dir, db_id)
    if os.path.isdir(folder):
        for f in os.listdir(folder):
            if f.endswith('.sqlite'):
                return os.path.join(folder, f)
    direct = os.path.join(db_dir, f"{db_id}.sqlite")
    if os.path.exists(direct):
        return direct
    return None

def get_db_schema(db_path: str) -> Dict[str, Dict[str, str]]:
    if not os.path.exists(db_path):
        return {}
    schema = {}
    try:
        import sqlite3
        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
            tables = [row[0] for row in cursor.fetchall()]
            for table in tables:
                safe_table = table.replace('"', '""')
                cursor.execute(f'PRAGMA table_info("{safe_table}");')
                schema[table] = {row[1]: row[2] for row in cursor.fetchall()}
    except Exception as e:
        logger.warning(f"Error reading {db_path}: {e}")
    return schema

def build_spider_dataframe(json_paths, db_dir: str) -> pd.DataFrame:
    if isinstance(json_paths, str):
        json_paths = [json_paths]
    dataset = []
    for path in json_paths:
        with open(path, 'r', encoding='utf-8') as f:
            dataset.extend(json.load(f))
    schema_cache = {}
    records = []
    for item in dataset:
        db_id = item['db_id']
        question = item.get('question', '')
        gold_query = item.get('query', item.get('SQL', ''))
        if db_id not in schema_cache:
            db_path = find_sqlite_path(db_dir, db_id)
            schema_cache[db_id] = get_db_schema(db_path) if db_path else {}
        records.append({
            "schema_json": json.dumps(schema_cache[db_id], ensure_ascii=False),
            "gold_query": gold_query,
            "question": question
        })
    return pd.DataFrame(records)

In [ ]:
def extract_classes(text: str) -> List[str]:
    classes = []
    text = text.lower()
    if " group by " in text:
        classes.append("GroupQ")
    if " order by " in text:
        classes.append("SortQ")
    if " join " in text:
        classes.append("JOINQ")
    if any(k in text for k in ["count(", "sum(", "avg(", "min(", "max("]):
        classes.append("AggregationQ")
    if " distinct " in text:
        classes.append("DistinctQ")
    if " having " in text:
        classes.append("HavingQ")
    if " where " in text:
        classes.append("WhereQ")
    if " between " in text or "interval" in text:
        classes.append("DateFilterQ")
    if " in " in text:
        classes.append("InListQ")
    if "null" in text:
        classes.append("NullCheckQ")
    if any(k in text for k in [" limit ", "top ", "offset "]):
        classes.append("LimitQ")
    return classes

In [ ]:
def schema_to_text(schema_dict: Dict) -> str:
    if not isinstance(schema_dict, dict):
        return ""
    parts = []
    for table, columns in schema_dict.items():
        if isinstance(columns, dict):
            cols = ', '.join([f"{col}({dtype})" for col, dtype in columns.items()])
        elif isinstance(columns, list):
            cols = ', '.join(str(c) for c in columns)
        else:
            cols = str(columns)
        parts.append(f"{table}: [{cols}]")
    return ' | '.join(parts)

def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df['schema_parsed'] = df['schema_json'].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    df['classes'] = df['gold_query'].apply(extract_classes)
    df['schema_text'] = df['schema_parsed'].apply(schema_to_text)
    df['input_text'] = df['question'] + ' [SCHEMA] ' + df['schema_text']
    return df

In [ ]:
def extract_classes(text: str) -> List[str]:
    classes = []
    text = text.lower()
    if " group by " in text:
        classes.append("GroupQ")
    if " order by " in text:
        classes.append("SortQ")
    if " join " in text:
        classes.append("JOINQ")
    if any(k in text for k in ["count(", "sum(", "avg(", "min(", "max("]):
        classes.append("AggregationQ")
    if " distinct " in text:
        classes.append("DistinctQ")
    if " having " in text:
        classes.append("HavingQ")
    if " where " in text:
        classes.append("WhereQ")
    if " between " in text or "interval" in text:
        classes.append("DateFilterQ")
    if " in " in text:
        classes.append("InListQ")
    if "null" in text:
        classes.append("NullCheckQ")
    if any(k in text for k in [" limit ", "top ", "offset "]):
        classes.append("LimitQ")
    return classes

In [ ]:
def schema_to_text(schema_dict: Dict) -> str:
    if not isinstance(schema_dict, dict):
        return ""
    parts = []
    for table, columns in schema_dict.items():
        if isinstance(columns, dict):
            cols = ', '.join([f"{col}({dtype})" for col, dtype in columns.items()])
        elif isinstance(columns, list):
            cols = ', '.join(str(c) for c in columns)
        else:
            cols = str(columns)
        parts.append(f"{table}: [{cols}]")
    return ' | '.join(parts)

def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df['schema_parsed'] = df['schema_json'].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    df['classes'] = df['gold_query'].apply(extract_classes)
    df['schema_text'] = df['schema_parsed'].apply(schema_to_text)
    df['input_text'] = df['question'] + ' [SCHEMA] ' + df['schema_text']
    return df

In [ ]:
df = build_spider_dataframe(
    [os.path.join(cfg.data_dir, "train_spider.json"),
     os.path.join(cfg.data_dir, "dev.json")],
    os.path.join(cfg.data_dir, "database")
)
df = preprocess_dataframe(df)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['classes'])

X_train_val, X_test, y_train_val, y_test = train_test_split(
    df['input_text'].values, y, test_size=cfg.test_size, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=cfg.val_size, random_state=SEED
)

test_df = pd.DataFrame({
    'input_text': X_test,
    'gold_labels': mlb.inverse_transform(y_test)
})
test_df.to_csv(os.path.join(cfg.output_dir, 'test_dataset.csv'), index=False)

baseline

In [ ]:
encoder = SentenceTransformer(cfg.sentence_model)
X_train_emb = encoder.encode(X_train.tolist(), show_progress_bar=True, batch_size=64)
X_val_emb = encoder.encode(X_val.tolist(), show_progress_bar=True, batch_size=64)
X_test_emb = encoder.encode(X_test.tolist(), show_progress_bar=True, batch_size=64)

clf = OneVsRestClassifier(
    LogisticRegression(max_iter=cfg.lr_max_iter, C=cfg.lr_c, class_weight='balanced', solver='lbfgs')
)
clf.fit(X_train_emb, y_train)

y_pred_baseline = clf.predict(X_test_emb)

baseline_metrics = {
    'hamming_loss': float(hamming_loss(y_test, y_pred_baseline)),
    'subset_accuracy': float(accuracy_score(y_test, y_pred_baseline)),
    'f1_micro': float(f1_score(y_test, y_pred_baseline, average='micro', zero_division=0)),
    'f1_macro': float(f1_score(y_test, y_pred_baseline, average='macro', zero_division=0)),
    'f1_samples': float(f1_score(y_test, y_pred_baseline, average='samples', zero_division=0)),
    'classification_report': classification_report(
        y_test, y_pred_baseline, target_names=mlb.classes_, output_dict=True, zero_division=0
    )
}

joblib.dump(clf, os.path.join(cfg.model_dir, 'baseline_clf.pkl'))
joblib.dump(encoder, os.path.join(cfg.model_dir, 'baseline_encoder.pkl'))
joblib.dump(mlb, os.path.join(cfg.model_dir, 'mlb.pkl'))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

['./artifacts/models/mlb.pkl']

In [ ]:
class TextMultilabelDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist() if isinstance(texts, np.ndarray) else texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt'
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.bert_model)
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.bert_model,
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

train_ds = TextMultilabelDataset(X_train, y_train, tokenizer, cfg.max_length)
val_ds = TextMultilabelDataset(X_val, y_val, tokenizer, cfg.max_length)
test_ds = TextMultilabelDataset(X_test, y_test, tokenizer, cfg.max_length)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        'hamming_loss': hamming_loss(labels, preds),
        'subset_accuracy': accuracy_score(labels, preds),
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_samples': f1_score(labels, preds, average='samples', zero_division=0),
    }

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(cfg.output_dir, "bert_model"),
    learning_rate=cfg.lr_bert,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size * 2,
    num_train_epochs=cfg.train_epochs,
    weight_decay=cfg.weight_decay,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Hamming Loss,Subset Accuracy,F1 Micro,F1 Macro,F1 Samples,Runtime,Samples Per Second,Steps Per Second
1,0.223802,0.208228,0.069525,0.464219,0.814547,0.505107,0.759213,6.495600,247.398000,7.851000
2,0.145171,0.141851,0.049103,0.611699,0.875965,0.626964,0.823824,9.472800,169.644000,5.384000
3,0.120040,0.122208,0.040165,0.692595,0.899004,0.693025,0.850569,7.106600,226.129000,7.176000
4,0.092877,0.109609,0.035809,0.718731,0.909325,0.701331,0.864073,6.974600,230.409000,7.312000
5,0.078032,0.105798,0.035809,0.717486,0.909429,0.703753,0.865101,6.622200,242.669000,7.701000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1510, training_loss=0.16145207254302424, metrics={'train_runtime': 374.1825, 'train_samples_per_second': 64.407, 'train_steps_per_second': 4.035, 'total_flos': 798244176307200.0, 'train_loss': 0.16145207254302424, 'epoch': 5.0})

In [ ]:
predictions = trainer.predict(test_ds)
logits, true_labels = predictions.predictions, predictions.label_ids

probs = 1 / (1 + np.exp(-logits))
pred_labels = (probs >= 0.5).astype(int)

bert_metrics = {
    'hamming_loss': float(hamming_loss(true_labels, pred_labels)),
    'subset_accuracy': float(accuracy_score(true_labels, pred_labels)),
    'f1_micro': float(f1_score(true_labels, pred_labels, average='micro', zero_division=0)),
    'f1_macro': float(f1_score(true_labels, pred_labels, average='macro', zero_division=0)),
    'f1_samples': float(f1_score(true_labels, pred_labels, average='samples', zero_division=0)),
    'classification_report': classification_report(
        true_labels, pred_labels, target_names=mlb.classes_, output_dict=True, zero_division=0
    )
}

trainer.save_model(os.path.join(cfg.model_dir, "bert_final"))
tokenizer.save_pretrained(os.path.join(cfg.model_dir, "bert_final"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./artifacts/models/bert_final/tokenizer_config.json',
 './artifacts/models/bert_final/tokenizer.json')

In [ ]:
artifacts = {
    'models': [
        os.path.join(cfg.model_dir, 'baseline_clf.pkl'),
        os.path.join(cfg.model_dir, 'baseline_encoder.pkl'),
        os.path.join(cfg.model_dir, 'bert_final'),
        os.path.join(cfg.model_dir, 'mlb.pkl')
    ],
    'data': [
        os.path.join(cfg.output_dir, 'test_dataset.csv'),
        os.path.join(cfg.output_dir, 'predictions.csv')
    ],
    'metrics': os.path.join(cfg.output_dir, 'results.json'),
    'config': os.path.join(cfg.output_dir, 'config.json'),
    'logs': os.path.join(cfg.output_dir, 'experiment.log')
}

with open(os.path.join(cfg.output_dir, 'artifacts.json'), 'w') as f:
    json.dump(artifacts, f, indent=2)

logger.info(f"Experiment completed. Artifacts saved to {cfg.output_dir}")

Дообучение на датасете BIRD

In [ ]:
from datasets import load_dataset

def build_bird_dataframe(split: str = "train") -> pd.DataFrame:
    bird_ds = load_dataset("Sudnya/bird-sql", split=split)
    records = []
    schema_cache = {}
    db_dir = os.path.join(cfg.data_dir, "database")

    for item in bird_ds:
        db_id = item.get('db_id', '')
        question = item.get('question', '')
        gold_query = item.get('SQL', '')

        if db_id and db_id not in schema_cache:
            db_path = find_sqlite_path(db_dir, db_id)
            schema_cache[db_id] = get_db_schema(db_path) if db_path else {}

        records.append({
            "schema_json": json.dumps(schema_cache.get(db_id, {}), ensure_ascii=False),
            "gold_query": gold_query,
            "question": question,
            "source": "bird"
        })

    return pd.DataFrame(records)

bird_df = build_bird_dataframe(split="train")

if not bird_df.empty:
    bird_df = preprocess_dataframe(bird_df)
    bird_classes = bird_df['classes'].tolist()
    y_bird = mlb.transform(bird_classes)
    X_bird = bird_df['input_text'].values

    X_bird_train, X_bird_val, y_bird_train, y_bird_val = train_test_split(
        X_bird, y_bird, test_size=cfg.val_size, random_state=SEED
    )

    bird_train_ds = TextMultilabelDataset(X_bird_train, y_bird_train, tokenizer, cfg.max_length)
    bird_val_ds = TextMultilabelDataset(X_bird_val, y_bird_val, tokenizer, cfg.max_length)

    model = AutoModelForSequenceClassification.from_pretrained(
        os.path.join(cfg.model_dir, "bert_final"),
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification"
    )

    bird_training_args = TrainingArguments(
        output_dir=os.path.join(cfg.output_dir, "bert_bird_finetuned"),
        learning_rate=cfg.lr_bert * 0.5,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * 2,
        num_train_epochs=15,
        weight_decay=cfg.weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=25,
        report_to="none",
        save_total_limit=1,
    )

    bird_trainer = Trainer(
        model=model,
        args=bird_training_args,
        train_dataset=bird_train_ds,
        eval_dataset=bird_val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    bird_trainer.train()
    bird_trainer.evaluate()
    bird_trainer.save_model(os.path.join(cfg.model_dir, "bert_bird_finetuned"))
    tokenizer.save_pretrained(os.path.join(cfg.model_dir, "bert_bird_finetuned"))

    artifacts['models'].append(os.path.join(cfg.model_dir, "bert_bird_finetuned"))

with open(os.path.join(cfg.output_dir, 'BIRD_artifacts.json'), 'w') as f:
    json.dump(artifacts, f, indent=2)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/204k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9428 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1534 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
import json
import os

metrics_keys = ['hamming_loss', 'subset_accuracy', 'f1_micro', 'f1_macro', 'f1_samples']
metric_labels = {
    'hamming_loss': 'Hamming Loss ↓',
    'subset_accuracy': 'Exact Match (Subset Acc) ↑',
    'f1_micro': 'F1 Micro ↑',
    'f1_macro': 'F1 Macro ↑',
    'f1_samples': 'F1 Samples ↑'
}

comparison = []
for k in metrics_keys:
    delta = bert_metrics[k] - baseline_metrics[k]
    comparison.append({
        'Метрика': metric_labels[k],
        'Baseline (LR + MiniLM)': round(baseline_metrics[k], 4),
        'DistilBERT (Spider → BIRD)': round(bert_metrics[k], 4),
        'Δ': round(delta, 4)
    })

df_metrics = pd.DataFrame(comparison)
display(df_metrics)

os.makedirs(cfg.output_dir, exist_ok=True)
df_metrics.to_csv(os.path.join(cfg.output_dir, 'metrics_comparison.csv'), index=False)

final_report = {
    'experiment_config': {
        'dataset': 'Spider + BIRD',
        'classes': mlb.classes_.tolist(),
        'seed': SEED
    },
    'baseline_metrics': baseline_metrics,
    'final_model_metrics': bert_metrics,
    'comparison_table': comparison
}

with open(os.path.join(cfg.output_dir, 'final_report.json'), 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False)



,Метрика,Baseline (LR + MiniLM),DistilBERT (Spider → BIRD),Δ
0,Hamming Loss ↓,0.1205,0.0397,-0.0808
1,Exact Match (Subset Acc) ↑,0.2862,0.6839,0.3976
2,F1 Micro ↑,0.7299,0.8961,0.1662
3,F1 Macro ↑,0.5888,0.6887,0.0999
4,F1 Samples ↑,0.6912,0.8529,0.1617


In [ ]:
!zip -r /content/artifacts.zip /content/artifacts

updating: content/artifacts/ (stored 0%)
updating: content/artifacts/bert_model/ (stored 0%)
updating: content/artifacts/bert_model/checkpoint-1510/ (stored 0%)
updating: content/artifacts/bert_model/checkpoint-1510/tokenizer_config.json (deflated 42%)
updating: content/artifacts/bert_model/checkpoint-1510/model.safetensors (deflated 8%)
updating: content/artifacts/bert_model/checkpoint-1510/optimizer.pt (deflated 37%)
updating: content/artifacts/bert_model/checkpoint-1510/trainer_state.json (deflated 74%)
updating: content/artifacts/bert_model/checkpoint-1510/training_args.bin (deflated 53%)
updating: content/artifacts/bert_model/checkpoint-1510/scheduler.pt (deflated 61%)
updating: content/artifacts/bert_model/checkpoint-1510/config.json (deflated 58%)
updating: content/artifacts/bert_model/checkpoint-1510/rng_state.pth (deflated 26%)
updating: content/artifacts/bert_model/checkpoint-1510/tokenizer.json (deflated 71%)
updating: content/artifacts/artifacts.json (deflated 59%)
updating